> **In this repository.** Checks how robust the BioCLIP 2 zero-shot result of notebook 14 is (unknown species, a larger vocabulary) and whether light training on the frozen features helps, on `val.csv` only. It was run on Google Colab against the team's Google Drive, so its paths point to Drive; the saved outputs are the record of what it produced. To re-run it from this repository, change the data path to the folder made by `src/invasive_clf/data/download_images.py` (see `data/README.md`).

# 15 — BioCLIP 2: how robust is the zero-shot result, and does a little training help?

Notebook 14 gave BioCLIP 2 zero-shot macro-F1 ≈ 0.97 on val, far above the fine-tuned runs. That run had an easy setting: BioCLIP only had to choose among the 441 species that are actually in the dataset. This notebook asks five questions, **on val only** (`test.csv` / `test_unseen.csv` are never read), reusing the embeddings notebook 14 saved, so no images are re-encoded:

**A. Open set (unknown species).** Remove a share of the background species from BioCLIP's vocabulary and see how often their images are flagged as invasive. This is the val-only stand-in for `test_unseen`, and it measures the false alarms that matter in the field.

**B. Bigger vocabulary.** Add up to ~4,000 other Desert Southwest species from the dataset's `classes.csv` (the candidate list of the final query) as possible answers, so BioCLIP can no longer win by elimination.

**C. A little training on frozen features.** Linear heads on the saved image embeddings (CPU, minutes): a random 40-class head trained like the template's frozen stage, a 441-species head, the same head **initialised from BioCLIP's text embeddings** (starts exactly at zero-shot), and a weight interpolation between zero-shot and the trained head (WiSE-FT).

**D. Open set with the full vocabulary.** Part A repeated with all extra species from B in the vocabulary.

**E. Better classifiers, same 40-output labels.** Logistic regression (the benchmark recipe), softmax logistic regression, a small network and k-nearest neighbours on the frozen embeddings.

All metrics use the template's code (copied unchanged), so every number is comparable with notebook 14 and with the benchmark runs. **Runtime:** CPU is enough (~10–15 min, most of it loading the model to encode the extra species names and fetching nothing else); a GPU makes the text encoding faster.

## 0. Environment

In [ ]:
import importlib.util, os, subprocess, sys
_required = {"torchmetrics": "torchmetrics", "open_clip": "open_clip_torch"}
_missing = [pkg for mod, pkg in _required.items() if importlib.util.find_spec(mod) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])
IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
import json, math, re, time, datetime, copy
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import torch
import torch.nn as nn
import torch.nn.functional as F
import open_clip
import torchmetrics
from torchmetrics.classification import MulticlassF1Score, MulticlassAccuracy, MulticlassConfusionMatrix
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve, roc_curve,
                             precision_score, recall_score, f1_score)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | open_clip {open_clip.__version__} | device {DEVICE}")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 9})
C_BLUE, C_ORANGE, C_GRAY, C_AQUA, C_YELLOW = "#2a78d6", "#eb6834", "#9a9995", "#1baf7a", "#eda100"

## 1. Settings and paths

In [ ]:
MODEL_ID = "hf-hub:imageomics/bioclip-2"
PROMPT = None                 # None = the prompt notebook 14 accepted (decisions.json); or "scientific" / "taxonomic" / "taxonomic_common"
TTA_HFLIP = True              # same as notebook 14: average the image and mirror-image embeddings
# A. open set
OPENSET_FRACS = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]   # share of background species removed from the vocabulary
OPENSET_SEEDS = 5
# B. bigger vocabulary (extra species from classes.csv, most-observed first). "scientific" prompts need no taxonomy lookup.
VOCAB_PROMPT = "scientific"
VOCAB_EXTRA = [0, 250, 500, 1000, 2000, "all"]
# C. linear probes on frozen embeddings
PROBE_EPOCHS = {"bce40_random": 40, "species_random": 40, "species_textinit": 15}
PROBE_LR = {"bce40_random": 1e-3, "species_random": 1e-3, "species_textinit": 1e-4}
PROBE_WD = 1e-4
BATCH = 256
SEED = 42
N_BOOT = 1000                 # paired bootstrap for the difference between two runs

TEAM_DIR = Path("/content/drive/MyDrive/fall-2026-DL-invasive-plants")
PERSONAL_RUNS = Path("/content/drive/MyDrive/my_modeling_runs")   # fallback when the team folder is read-only
_candidates = [TEAM_DIR / "downloaded_data/inat_dataset_final",
               Path("/content/drive/Shareddrives/fall-2026-DL-invasive-plants/downloaded_data/inat_dataset_final"),
               Path("../../downloaded_data/inat_dataset_final"), Path("downloaded_data/inat_dataset_final")]
DATA_ROOT = next((p for p in _candidates if (p / "val.csv").exists()), None)
assert DATA_ROOT is not None, "dataset folder not found (add the team folder to My Drive as a shortcut)"
TEAM_RUNS = DATA_ROOT.parent.parent / "modeling_runs"

# Outputs of notebook 14 (team folder first, then the personal fallback)
ZS_DIR = next((d for d in [TEAM_RUNS / "bioclip2_zeroshot", PERSONAL_RUNS / "bioclip2_zeroshot"]
               if (d / "embeddings_val.npz").exists()), None)
assert ZS_DIR is not None, "run notebook 14 first: embeddings_val.npz not found"
assert (ZS_DIR / "embeddings_train.npz").exists(), "notebook 14 was run with EMBED_TRAIN=False; re-run it with EMBED_TRAIN=True"

def _writable(d: Path) -> bool:
    try:
        d.mkdir(parents=True, exist_ok=True); p = d / ".write_test"; p.write_text("ok"); p.unlink(); return True
    except OSError:
        return False

OUT_PROBE = TEAM_RUNS / "bioclip2_probe"
OUT_ROBUST = ZS_DIR / "robustness"
if not (_writable(OUT_PROBE) and _writable(OUT_ROBUST)):
    OUT_PROBE, OUT_ROBUST = PERSONAL_RUNS / "bioclip2_probe", PERSONAL_RUNS / "bioclip2_zeroshot" / "robustness"
    assert _writable(OUT_PROBE) and _writable(OUT_ROBUST)
    print("team folder is read-only for this account: writing to your own folder")

if PROMPT is None:
    PROMPT = next(d["name"] for d in json.load(open(ZS_DIR / "decisions.json")) if d["decision"] == "accepted")
ZS_METRICS = json.load(open(ZS_DIR / PROMPT / "metrics_final.json"))
T_ZS = ZS_METRICS["threshold"]                                 # threshold notebook 14 tuned on val, kept fixed below
print("DATA_ROOT:", DATA_ROOT); print("notebook-04 outputs:", ZS_DIR); print("prompt:", PROMPT, "| zero-shot threshold:", T_ZS)
print("writing to:", OUT_ROBUST, "and", OUT_PROBE)

## 2. Data, labels and the saved embeddings

In [ ]:
NONE_LABEL = "not_invasive"
train_df = pd.read_csv(DATA_ROOT / "train.csv")
val_df = pd.read_csv(DATA_ROOT / "val.csv")
classes = pd.read_csv(DATA_ROOT / "classes.csv")
INVASIVE_CLASSES = sorted(c for c in train_df["label"].unique() if c != NONE_LABEL)
N_INVASIVE = len(INVASIVE_CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(INVASIVE_CLASSES)}; CLASS_TO_IDX[NONE_LABEL] = N_INVASIVE
ALL_LABELS = INVASIVE_CLASSES + [NONE_LABEL]

zv = np.load(ZS_DIR / "embeddings_val.npz", allow_pickle=True)
zt = np.load(ZS_DIR / "embeddings_train.npz", allow_pickle=True)
VOCAB_IDS = zv["vocab_taxon_id"].astype(int)                      # vocabulary order used in notebook 14
VIDX = {t: i for i, t in enumerate(VOCAB_IDS)}
lab = pd.concat([train_df, val_df]).drop_duplicates("taxon_id").set_index("taxon_id")
vocab = pd.DataFrame({"taxon_id": VOCAB_IDS, "label": lab.loc[VOCAB_IDS, "label"].values,
                      "scientific_name": lab.loc[VOCAB_IDS, "scientific_name"].values, "group": lab.loc[VOCAB_IDS, "group"].values})
assert (vocab.label.values == zv["vocab_label"].astype(str)).all(), "vocabulary does not match notebook 14"
vocab["y"] = vocab.label.map(CLASS_TO_IDX)
_inv = np.flatnonzero(vocab.y.to_numpy() < N_INVASIVE)
M = np.zeros((len(vocab), N_INVASIVE), dtype=np.float32); M[_inv, vocab.y.to_numpy()[_inv]] = 1
BG = np.flatnonzero(vocab.y.to_numpy() == N_INVASIVE)              # background classes in the vocabulary

def aligned(df, z):
    # rows of df in the order of the saved embeddings (images that failed to load in notebook 14 are dropped)
    pos = pd.Series(np.arange(len(z["observation_id"])), index=z["observation_id"].astype(np.int64))
    df = df[df.observation_id.isin(pos.index)].copy()
    order = pos.loc[df.observation_id].to_numpy()
    plain = torch.tensor(z["image_emb"][order].astype(np.float32)); flip = torch.tensor(z["image_emb_flip"][order].astype(np.float32))
    df = df.reset_index(drop=True)
    df["y"] = df.label.map(CLASS_TO_IDX); df["v"] = df.taxon_id.map(VIDX)
    return df, F.normalize(plain, dim=-1), F.normalize(flip, dim=-1)

val_df, val_plain, val_flip = aligned(val_df, zv)
train_df, tr_plain, tr_flip = aligned(train_df, zt)
val_emb = F.normalize(val_plain + val_flip, dim=-1) if TTA_HFLIP else val_plain
y_val, v_val = val_df.y.to_numpy(), val_df.v.to_numpy()
TEXT = {k[5:]: F.normalize(torch.tensor(zv[k].astype(np.float32)), dim=-1) for k in zv.files if k.startswith("text_")}
print(f"vocabulary {len(vocab)} classes ({len(BG)} background) | val {len(val_df)} | train {len(train_df)} | prompts saved: {list(TEXT)}")

## 3. Model (only for its temperature and to encode the extra species names) and the template metrics

In [ ]:
clip = open_clip.create_model(MODEL_ID).to(DEVICE).eval()
tokenizer = open_clip.get_tokenizer(MODEL_ID)
LOGIT_SCALE = clip.logit_scale.exp().item()
print("logit scale:", round(LOGIT_SCALE, 2))

@torch.no_grad()
def encode_texts(texts, bs=256):
    out = []
    for i in range(0, len(texts), bs):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            e = clip.encode_text(tokenizer(texts[i:i + bs]).to(DEVICE)).float()
        out.append(F.normalize(e, dim=-1).cpu())
    return torch.cat(out)

In [ ]:
# ---------- copied unchanged from the fine-tuning template (notebooks 05 to 13), Section 5 ----------
def predict_not_invasive(probs: np.ndarray, t: float) -> np.ndarray:
    # Species when the strongest invasive probability clears t, otherwise not_invasive.
    pred = probs.argmax(1)
    pred[probs.max(1) < t] = N_INVASIVE
    return pred


def macro_f1_invasive(y: np.ndarray, pred: np.ndarray) -> float:
    f1 = MulticlassF1Score(num_classes=N_INVASIVE + 1, average=None)(torch.as_tensor(pred), torch.as_tensor(y))
    return float(f1[:N_INVASIVE].mean())


def tune_threshold(probs, y, grid):
    # Sweep the grid; return the best t plus the full sweep for plotting.
    sweep = pd.DataFrame({"threshold": grid})
    sweep["macro_f1_invasive"] = [macro_f1_invasive(y, predict_not_invasive(probs, t)) for t in grid]
    sweep["top1_acc_all"] = [float((predict_not_invasive(probs, t) == y).mean()) for t in grid]
    best = sweep.loc[sweep["macro_f1_invasive"].idxmax()]
    return float(best["threshold"]), sweep


def compute_metrics(probs: np.ndarray, y: np.ndarray, threshold: float | None, grid) -> dict:
    probs = np.asarray(probs, dtype=np.float64); y = np.asarray(y)
    is_inv = y < N_INVASIVE
    onehot = np.eye(N_INVASIVE + 1)[y][:, :N_INVASIVE]              # (N, N_INVASIVE) binary targets

    m = {}
    # threshold-free, ranking metrics over the invasive classes
    per_class_ap = np.array([average_precision_score(onehot[:, c], probs[:, c]) if onehot[:, c].any() else np.nan
                             for c in range(N_INVASIVE)])
    m["macro_ap_invasive"] = float(np.nanmean(per_class_ap))
    if is_inv.any():
        rank = (-probs[is_inv]).argsort(1)
        m["top1_recall_invasive"] = float((rank[:, 0] == y[is_inv]).mean())
        m["top3_recall_invasive"] = float((rank[:, :3] == y[is_inv, None]).any(1).mean())
    # threshold
    sweep = None
    if threshold is None:
        threshold, sweep = tune_threshold(probs, y, grid)
    m["threshold"] = float(threshold)
    pred = predict_not_invasive(probs, threshold)
    # 41-way metrics at the fixed threshold
    f1_per_class = MulticlassF1Score(num_classes=N_INVASIVE + 1, average=None)(torch.as_tensor(pred), torch.as_tensor(y)).numpy()
    m["macro_f1_invasive"] = float(f1_per_class[:N_INVASIVE].mean())
    m["macro_f1_all41"] = float(f1_per_class.mean())
    m["top1_acc_all"] = float(MulticlassAccuracy(num_classes=N_INVASIVE + 1, average="micro")(torch.as_tensor(pred), torch.as_tensor(y)))
    # binary invasive detection
    score_inv = probs.max(1); pred_inv = pred < N_INVASIVE
    m["bin_precision_invasive"] = precision_score(is_inv, pred_inv, zero_division=0)
    m["bin_recall_invasive"] = recall_score(is_inv, pred_inv, zero_division=0)
    m["bin_f1_invasive"] = f1_score(is_inv, pred_inv, zero_division=0)
    m["bin_ap_invasive"] = average_precision_score(is_inv, score_inv)
    m["bin_auroc_invasive"] = roc_auc_score(is_inv, score_inv) if 0 < is_inv.sum() < len(y) else np.nan
    # the not_invasive side
    m["noninv_precision"] = precision_score(~is_inv, ~pred_inv, zero_division=0)
    m["noninv_recall"] = recall_score(~is_inv, ~pred_inv, zero_division=0)
    m["noninv_auroc"] = roc_auc_score(~is_inv, 1 - score_inv) if 0 < is_inv.sum() < len(y) else np.nan
    # arrays
    m["per_class_f1"] = f1_per_class.tolist()
    m["per_class_ap"] = per_class_ap.tolist()
    m["confusion_matrix"] = MulticlassConfusionMatrix(num_classes=N_INVASIVE + 1)(torch.as_tensor(pred), torch.as_tensor(y)).numpy().tolist()
    m["_sweep"] = sweep; m["_pred"] = pred
    return m

SCALAR_METRICS = ["macro_f1_invasive", "top1_acc_all", "macro_ap_invasive", "top1_recall_invasive", "top3_recall_invasive",
                  "threshold", "bin_precision_invasive", "bin_recall_invasive", "bin_f1_invasive", "bin_ap_invasive",
                  "bin_auroc_invasive", "noninv_precision", "noninv_recall", "noninv_auroc", "macro_f1_all41"]
KEY_METRICS = ["macro_f1_invasive", "top1_acc_all", "macro_ap_invasive", "top1_recall_invasive", "top3_recall_invasive",
               "bin_precision_invasive", "bin_recall_invasive", "bin_auroc_invasive", "noninv_auroc", "threshold"]

def scalars(m: dict) -> dict:
    return {k: m[k] for k in SCALAR_METRICS if k in m}

THRESHOLD_GRID = [round(t, 3) for t in np.linspace(0.01, 0.99, 99)]    # the template's threshold_grid

In [ ]:
# ---------- helpers shared by A, B and C ----------
def probs_from_text(img, txt, Mmat):
    p = torch.softmax(LOGIT_SCALE * img @ txt.T, dim=1).numpy()
    return p, p @ Mmat

def fast_macro_f1(y, pred, n=N_INVASIVE + 1):
    # same quantity as macro_f1_invasive (torchmetrics), via bincount: used inside the bootstrap
    cm = np.bincount(y * n + pred, minlength=n * n).reshape(n, n)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    f1 = np.divide(2 * tp, 2 * tp + fp + fn, out=np.zeros(n), where=(2 * tp + fp + fn) > 0)
    return f1[:N_INVASIVE].mean()

def paired_bootstrap(y, pred_a, pred_b, n_boot=N_BOOT, seed=0):
    # 95% interval of macro-F1(b) - macro-F1(a), resampling val images
    rng = np.random.default_rng(seed); d = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        d.append(fast_macro_f1(y[i], pred_b[i]) - fast_macro_f1(y[i], pred_a[i]))
    return np.percentile(d, [2.5, 50, 97.5])

# Reproduce notebook 14 from the saved embeddings (sanity check)
p_vocab_zs, p_inv_zs = probs_from_text(val_emb, TEXT[PROMPT], M)
m_zs = compute_metrics(p_inv_zs, y_val, None, THRESHOLD_GRID)
print(f"zero-shot from saved embeddings: macro-F1 {m_zs['macro_f1_invasive']:.4f} (notebook 14: {ZS_METRICS['macro_f1_invasive']:.4f}), "
      f"top-1 {m_zs['top1_acc_all']:.4f} (04: {ZS_METRICS['top1_acc_all']:.4f})")
assert abs(m_zs["macro_f1_invasive"] - ZS_METRICS["macro_f1_invasive"]) < 2e-3, "does not reproduce notebook 14"

## A. Open set: how often are *unknown* species flagged as invasive?

For each share *f* of background species (and several random draws), those species are deleted from the vocabulary; their val images stay. They can no longer be named correctly, so BioCLIP has to put them somewhere: on another background species (harmless) or on an invasive class (a false alarm). The threshold is **fixed** at notebook 14's value, as it would be in deployment; the threshold-free lookup rule is shown too. *f* = 1 is the extreme case of a vocabulary with only the 40 invasive classes.

In [ ]:
rows_open, absorbed = [], {}
is_inv_val = y_val < N_INVASIVE
for f in OPENSET_FRACS:
    for s in (range(OPENSET_SEEDS) if 0 < f < 1 else [0]):
        rng = np.random.default_rng(s)
        removed = rng.choice(BG, size=int(round(f * len(BG))), replace=False) if f > 0 else np.array([], dtype=int)
        keep = np.setdiff1d(np.arange(len(vocab)), removed)
        p_voc, p_inv = probs_from_text(val_emb, TEXT[PROMPT][keep], M[keep])
        pred = predict_not_invasive(p_inv, T_ZS)
        lookup = vocab.y.to_numpy()[keep][p_voc.argmax(1)]
        unseen = np.isin(v_val, removed); seen_bg = ~is_inv_val & ~unseen
        flagged, flagged_l = pred < N_INVASIVE, lookup < N_INVASIVE
        rows_open.append({"frac_removed": f, "seed": s, "n_unseen_images": int(unseen.sum()),
                          "false_alarm_unseen": flagged[unseen].mean() if unseen.any() else np.nan,
                          "false_alarm_seen": flagged[seen_bg].mean() if seen_bg.any() else np.nan,
                          "invasive_recall": flagged[is_inv_val].mean(),
                          "macro_f1_invasive": macro_f1_invasive(y_val, pred), "top1_acc_all": float((pred == y_val).mean()),
                          "lookup_false_alarm_unseen": flagged_l[unseen].mean() if unseen.any() else np.nan,
                          "lookup_macro_f1_invasive": macro_f1_invasive(y_val, lookup)})
        if f in (0.5, 1.0):
            hit = pd.Series(np.array(ALL_LABELS)[pred[unseen & flagged]]).value_counts()
            absorbed[f] = absorbed.get(f, pd.Series(dtype=float)).add(hit, fill_value=0)
openset = pd.DataFrame(rows_open)
openset.to_csv(OUT_ROBUST / "openset_leave_out.csv", index=False)
open_summary = openset.groupby("frac_removed").agg(["mean", "std"]).drop(columns="seed")
display(open_summary[["false_alarm_unseen", "false_alarm_seen", "invasive_recall", "macro_f1_invasive", "top1_acc_all",
                      "lookup_false_alarm_unseen"]].round(4))

In [ ]:
g = openset.groupby("frac_removed")
mean, std = g.mean(numeric_only=True), g.std(numeric_only=True).fillna(0)
x = mean.index.to_numpy() * 100
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
ax = axes[0]
for col, c, lab_ in [("false_alarm_unseen", C_ORANGE, "unknown species flagged invasive"),
                     ("false_alarm_seen", C_BLUE, "known background species flagged invasive")]:
    ax.errorbar(x, mean[col] * 100, yerr=std[col] * 100, color=c, lw=2, marker="o", ms=5, capsize=3, label=lab_)
ax.set_xlabel("background species removed from the vocabulary (%)"); ax.set_ylabel("false-alarm rate (%)"); ax.set_ylim(bottom=0)
ax.set_title(f"open set, fixed threshold t = {T_ZS:.2f}"); ax.legend(fontsize=7, frameon=False)
ax = axes[1]
for col, c, lab_ in [("macro_f1_invasive", C_BLUE, "macro-F1 (invasive, primary)"), ("top1_acc_all", C_ORANGE, "top-1 accuracy (41, KPI)"),
                     ("invasive_recall", C_AQUA, "invasive recall")]:
    ax.errorbar(x, mean[col], yerr=std[col], color=c, lw=2, marker="o", ms=5, capsize=3, label=lab_)
ax.set_ylim(0, 1.02); ax.set_xlabel("background species removed from the vocabulary (%)")
ax.set_title("overall val metrics as the vocabulary shrinks"); ax.legend(fontsize=7, frameon=False, loc="lower left")
fig.suptitle(f"BioCLIP 2 zero-shot ({PROMPT}): unknown species, val only (mean ± sd over {OPENSET_SEEDS} random draws)", fontsize=9)
fig.tight_layout(); fig.savefig(OUT_ROBUST / "openset_leave_out.png"); plt.show()

# Which invasive classes absorb unknown species?
for f, s in absorbed.items():
    s = (s / (OPENSET_SEEDS if f < 1 else 1)).sort_values().tail(12)
    if s.empty: continue
    fig, ax = plt.subplots(figsize=(6.5, 0.3 * len(s) + 1.2))
    ax.barh(s.index, s.values, color=C_ORANGE, height=0.6); ax.set_xlabel("unknown-species val images flagged as this class (per draw)")
    ax.tick_params(axis="y", labelsize=7); ax.set_title(f"where unknown species go, {int(f * 100)}% of background removed", fontsize=8)
    fig.tight_layout(); fig.savefig(OUT_ROBUST / f"openset_absorbers_{int(f * 100)}.png"); plt.show()

## B. Bigger vocabulary: BioCLIP has to choose among thousands of regional species

Extra classes come from the dataset's `classes.csv` (all Desert Southwest candidate classes of the final query), excluding: classes already in the vocabulary, any class sharing a member taxon with them (so no species can steal probability from an invasive complex), and the 30 `test_unseen` hold-out classes (so that test stays a test of unknown species). Extras are added most-observed first and all count as `not_invasive`. Prompts are `VOCAB_PROMPT` for every class, including the original 441, so the comparison is like for like.

In [ ]:
def split_ids(s):
    return [int(x) for x in re.split(r"[;|]", str(s)) if x.strip().isdigit()]
in_vocab_members = set(VOCAB_IDS)
for r in classes[classes.class_key.isin(VOCAB_IDS)].itertuples():
    in_vocab_members |= set(split_ids(r.member_taxon_ids))
cand = classes[~classes.class_key.isin(VOCAB_IDS) & ~classes.holdout_unseen.astype(bool)
               & classes.class_rank.isin(["species", "complex"])].copy()
cand = cand[~cand.member_taxon_ids.map(lambda s: bool(set(split_ids(s)) & in_vocab_members)) & ~cand.class_key.isin(in_vocab_members)]
cand = cand.drop_duplicates("class_key").sort_values("n_window", ascending=False).reset_index(drop=True)
print(f"{len(cand)} extra classes available; groups: {cand.group.value_counts().to_dict()}")
if (cand.group == "invasive").any():
    print("note: some extras are invasive-listed classes that are not among the 40 (dropped for too few images); they count as not_invasive here")

if VOCAB_PROMPT != "scientific":
    raise NotImplementedError("set VOCAB_PROMPT='scientific' (taxonomic prompts for ~4,000 extra species would need ~140 iNaturalist API calls)")
def names(r):
    ns = [n.strip() for n in re.split(r"[;|]", str(r.member_names)) if n.strip()]
    return ns or [r.class_name]
t0 = time.time()
base_txt = TEXT["scientific"]
per = [[f"a photo of {n}" for n in names(r)] for r in cand.itertuples()]
flat = encode_texts([p for ps in per for p in ps])
extra_txt = F.normalize(torch.stack([c.mean(0) for c in torch.split(flat, [len(ps) for ps in per])]), dim=-1)
print(f"encoded {len(flat)} extra prompts in {time.time() - t0:.0f}s")

In [ ]:
T_SCI = json.load(open(ZS_DIR / "scientific" / "metrics_final.json"))["threshold"]
rows_vocab = []
for n in sorted({len(cand) if n == "all" else min(n, len(cand)) for n in VOCAB_EXTRA}):
    txt = torch.cat([base_txt, extra_txt[:n]]); Mx = np.vstack([M, np.zeros((n, N_INVASIVE), np.float32)])
    p_voc, p_inv = probs_from_text(val_emb, txt, Mx)
    pred = predict_not_invasive(p_inv, T_SCI)
    m_ret = compute_metrics(p_inv, y_val, None, THRESHOLD_GRID)              # threshold re-tuned, for reference
    top1 = p_voc.argmax(1)
    rows_vocab.append({"n_extra": n, "vocab_size": len(txt),
                       "macro_f1_invasive": macro_f1_invasive(y_val, pred), "top1_acc_all": float((pred == y_val).mean()),
                       "invasive_recall": (pred < N_INVASIVE)[is_inv_val].mean(), "false_alarm": (pred < N_INVASIVE)[~is_inv_val].mean(),
                       "species_top1": float((top1 == v_val).mean()), "predicted_an_extra_species": float((top1 >= len(vocab)).mean()),
                       "macro_f1_retuned": m_ret["macro_f1_invasive"], "threshold_retuned": m_ret["threshold"]})
vocab_sweep = pd.DataFrame(rows_vocab)
vocab_sweep.to_csv(OUT_ROBUST / "vocabulary_size_sweep.csv", index=False)
display(vocab_sweep.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
ax = axes[0]
for col, c, lab_ in [("macro_f1_invasive", C_BLUE, "macro-F1 (invasive, primary)"), ("top1_acc_all", C_ORANGE, "top-1 accuracy (41, KPI)"),
                     ("species_top1", C_AQUA, "species top-1 (val species)"), ("invasive_recall", C_YELLOW, "invasive recall")]:
    ax.plot(vocab_sweep.vocab_size, vocab_sweep[col], color=c, lw=2, marker="o", ms=5, label=lab_)
ax.set_xscale("log"); ax.set_ylim(0, 1.02); ax.set_xlabel("vocabulary size (classes BioCLIP can choose from, log scale)")
ax.set_title(f"scientific-name prompts, fixed threshold t = {T_SCI:.2f}"); ax.legend(fontsize=7, frameon=False, loc="lower left")
ax = axes[1]
ax.plot(vocab_sweep.vocab_size, vocab_sweep.false_alarm * 100, color=C_ORANGE, lw=2, marker="o", ms=5, label="non-invasives flagged invasive")
ax.plot(vocab_sweep.vocab_size, vocab_sweep.predicted_an_extra_species * 100, color=C_BLUE, lw=2, marker="o", ms=5,
        label="val images named as an extra (absent) species")
ax.set_xscale("log"); ax.set_xlabel("vocabulary size (log scale)"); ax.set_ylabel("% of val images")
ax.set_title("false alarms and pulls towards absent species"); ax.legend(fontsize=7, frameon=False)
fig.suptitle("BioCLIP 2 zero-shot with a larger regional vocabulary (val only)", fontsize=9); fig.tight_layout()
fig.savefig(OUT_ROBUST / "vocabulary_size_sweep.png"); plt.show()

## C. Linear heads on the frozen BioCLIP 2 embeddings

Training data: the saved train embeddings, each image used twice (as is and mirrored, the frozen-feature version of flip augmentation). Every epoch is scored on val with the template's metrics and the best epoch by macro-F1 is kept, exactly as the fine-tuning notebooks (05 to 13) keep `best.pt` (so, like there, the best-epoch number is slightly optimistic).

| variant | head | loss | start |
|---|---|---|---|
| `bce40_random` | 40 logits, one per invasive class | multi-label BCE (template) | random (Xavier) |
| `species_random` | one logit per vocabulary species (441) | cross-entropy on species | random |
| `species_textinit` | the same, cosine classifier with BioCLIP's temperature | cross-entropy on species | **BioCLIP text embeddings** (epoch 0 = zero-shot) |
| `wise_alpha_*` | interpolation of the zero-shot and the trained text-init weights | – | – |

Species heads are turned into the template's 40 invasive probabilities exactly like zero-shot (sum of the species probabilities of each class).

In [ ]:
torch.manual_seed(SEED); np.random.seed(SEED)
Xtr = torch.cat([tr_plain, tr_flip]); ytr_v = torch.tensor(np.tile(train_df.v.to_numpy(), 2)); ytr_y = torch.tensor(np.tile(train_df.y.to_numpy(), 2))
target40 = torch.zeros(len(ytr_y), N_INVASIVE); target40[ytr_y < N_INVASIVE, ytr_y[ytr_y < N_INVASIVE]] = 1
print(f"train samples (with mirrors): {len(Xtr)}")

class LinearHead(nn.Module):
    def __init__(self, d, k):
        super().__init__(); self.fc = nn.Linear(d, k); nn.init.xavier_uniform_(self.fc.weight); nn.init.zeros_(self.fc.bias)
    def forward(self, x): return self.fc(x)

class CosineHead(nn.Module):
    # logits = scale * <x, w_c / |w_c|> + b_c ; with W = text embeddings and b = 0 this is exactly zero-shot BioCLIP
    def __init__(self, W, scale):
        super().__init__(); self.W = nn.Parameter(W.clone()); self.b = nn.Parameter(torch.zeros(len(W))); self.scale = scale
    def forward(self, x): return self.scale * x @ F.normalize(self.W, dim=-1).T + self.b

def val_probs(head, kind):
    head.eval()
    with torch.no_grad():
        z = head(val_emb)
    if kind == "bce40_random":
        return torch.sigmoid(z).numpy(), None
    p_voc = torch.softmax(z, 1).numpy()
    return p_voc @ M, p_voc

def train_probe(kind):
    torch.manual_seed(SEED)
    head = LinearHead(Xtr.shape[1], N_INVASIVE if kind == "bce40_random" else len(vocab)) if kind != "species_textinit" \
           else CosineHead(TEXT[PROMPT], LOGIT_SCALE)
    opt = torch.optim.AdamW(head.parameters(), lr=PROBE_LR[kind], weight_decay=PROBE_WD)
    hist, best = [], (-1, None, None)
    for ep in range(PROBE_EPOCHS[kind] + 1):            # epoch 0 = before training
        if ep > 0:
            head.train(); perm = torch.randperm(len(Xtr)); losses = []
            for i in range(0, len(Xtr), BATCH):
                b = perm[i:i + BATCH]; z = head(Xtr[b])
                loss = F.binary_cross_entropy_with_logits(z, target40[b]) if kind == "bce40_random" else F.cross_entropy(z, ytr_v[b])
                opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
        p_inv, p_voc = val_probs(head, kind)
        m = compute_metrics(p_inv, y_val, None, THRESHOLD_GRID)
        hist.append({"epoch": ep, "train_loss": float(np.mean(losses)) if ep else np.nan, **scalars(m)})
        if m["macro_f1_invasive"] > best[0]:
            best = (m["macro_f1_invasive"], copy.deepcopy(head.state_dict()), ep)
    head.load_state_dict(best[1])
    return head, pd.DataFrame(hist), best[2]

probes = {}
for kind in ["bce40_random", "species_random", "species_textinit"]:
    t0 = time.time(); head, hist, best_ep = train_probe(kind)
    probes[kind] = {"head": head, "hist": hist, "best_epoch": best_ep}
    print(f"{kind:>17}: best epoch {best_ep:>2} | macro-F1 {hist.macro_f1_invasive.max():.4f} | {time.time() - t0:.0f}s")

In [ ]:
# WiSE-FT for the head: interpolate the zero-shot text weights and the trained text-init weights (trained for all epochs)
torch.manual_seed(SEED)
full = CosineHead(TEXT[PROMPT], LOGIT_SCALE); opt = torch.optim.AdamW(full.parameters(), lr=PROBE_LR["species_textinit"], weight_decay=PROBE_WD)
for ep in range(PROBE_EPOCHS["species_textinit"]):
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), BATCH):
        b = perm[i:i + BATCH]; loss = F.cross_entropy(full(Xtr[b]), ytr_v[b]); opt.zero_grad(); loss.backward(); opt.step()
W0, W1, b1 = F.normalize(TEXT[PROMPT], dim=-1), F.normalize(full.W.detach(), dim=-1), full.b.detach()
rows_w = []; wise_probs = {}
for a in np.round(np.linspace(0, 1, 11), 1):
    h = CosineHead(F.normalize((1 - a) * W0 + a * W1, dim=-1), LOGIT_SCALE); h.b.data = a * b1
    p_inv, p_voc = val_probs(h, "species_textinit"); m = compute_metrics(p_inv, y_val, None, THRESHOLD_GRID)
    rows_w.append({"alpha": a, **scalars(m)}); wise_probs[a] = (p_inv, p_voc, m)
wise = pd.DataFrame(rows_w)
display(wise[["alpha", "macro_f1_invasive", "top1_acc_all", "bin_recall_invasive", "bin_precision_invasive", "threshold"]].round(4))
A_BEST = float(wise.loc[wise.macro_f1_invasive.idxmax(), "alpha"])
print("best alpha on val:", A_BEST, "(alpha 0 = zero-shot, 1 = fully trained head)")

### Save every variant like a template run and compare

In [ ]:
def save_run(name, p_inv, p_voc, m, extra=None, hist=None):
    d = OUT_PROBE / name; d.mkdir(parents=True, exist_ok=True)
    out = {"model": "bioclip2_probe", "iteration": name, "epochs": 0 if hist is None else int(hist.epoch.max()),
           "tta_hflip": TTA_HFLIP, **scalars(m), **(extra or {}), "per_class_f1": dict(zip(ALL_LABELS, m["per_class_f1"])),
           "per_class_ap": dict(zip(INVASIVE_CLASSES, m["per_class_ap"])), "labels": ALL_LABELS, "confusion_matrix": m["confusion_matrix"]}
    json.dump(out, open(d / "metrics_final.json", "w"), indent=1, default=float)
    if hist is not None: hist.to_csv(d / "history.csv", index=False)
    np.save(d / "val_probs_invasive.npy", p_inv.astype(np.float32))
    pd.DataFrame({"observation_id": val_df.observation_id, "label": val_df.label, "group": val_df.group, "scientific_name": val_df.scientific_name,
                  "pred": [ALL_LABELS[i] for i in m["_pred"]], "max_prob": p_inv.max(1)}).to_csv(d / "val_predictions.csv", index=False)

results = {"zero_shot (" + PROMPT + ")": (p_inv_zs, m_zs)}
for kind, r in probes.items():
    p_inv, p_voc = val_probs(r["head"], kind); m = compute_metrics(p_inv, y_val, None, THRESHOLD_GRID)
    save_run(kind, p_inv, p_voc, m, {"best_epoch": r["best_epoch"]}, r["hist"]); results[kind] = (p_inv, m)
p_inv, p_voc, m = wise_probs[A_BEST]; save_run(f"wise_alpha_{A_BEST}", p_inv, p_voc, m, {"alpha": A_BEST}); results[f"wise_alpha_{A_BEST}"] = (p_inv, m)
wise.to_csv(OUT_PROBE / "wise_sweep.csv", index=False)

cmp_rows = []
for name, (p, m) in results.items():
    lo, mid, hi = paired_bootstrap(y_val, m_zs["_pred"], m["_pred"]) if not name.startswith("zero_shot") else (0, 0, 0)
    cmp_rows.append({"run": name, **{k: m[k] for k in ["macro_f1_invasive", "top1_acc_all", "bin_recall_invasive", "bin_precision_invasive", "threshold"]},
                     "delta_vs_zero_shot": m["macro_f1_invasive"] - m_zs["macro_f1_invasive"], "delta_95ci_low": lo, "delta_95ci_high": hi})
cmp = pd.DataFrame(cmp_rows).set_index("run"); cmp.to_csv(OUT_PROBE / "comparison.csv")
display(cmp.round(4))
print("An interval that contains 0 means the difference from zero-shot is within val noise.")

### Plots for C

In [ ]:
# ---------- copied unchanged from the fine-tuning template (notebooks 05 to 13), Section 5b ----------
def plot_threshold_sweep(sweep: pd.DataFrame, t: float, out_dir: Path, title=""):
    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.plot(sweep["threshold"], sweep["macro_f1_invasive"], color=C_BLUE, lw=2, label="macro-F1 (invasive)")
    ax.plot(sweep["threshold"], sweep["top1_acc_all"], color=C_ORANGE, lw=2, label="top-1 accuracy (41)")
    ax.axvline(t, color=C_GRAY, ls="--", lw=1); ax.text(t, 0.02, f" t = {t:.2f}", fontsize=8)
    ax.set_xlabel("invasive threshold on max class probability"); ax.set_ylim(0, 1); ax.legend(fontsize=8)
    ax.set_title(title or "threshold sweep (validation)", fontsize=8); fig.tight_layout(); fig.savefig(out_dir / "threshold_sweep.png"); plt.show()


def plot_pr_roc(probs, y, m: dict, out_dir: Path, title=""):
    is_inv = y < N_INVASIVE; score = probs.max(1); t = m["threshold"]
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
    # binary PR curve: invasive vs not
    p, r, th = precision_recall_curve(is_inv, score)
    ax = axes[0]; ax.plot(r, p, color=C_BLUE, lw=2)
    ax.scatter([m["bin_recall_invasive"]], [m["bin_precision_invasive"]], color=C_ORANGE, s=40, zorder=3, label=f"operating point, t={t:.2f}")
    ax.axhline(is_inv.mean(), color=C_GRAY, ls=":", lw=1, label=f"prevalence {is_inv.mean():.2f}")
    ax.set_xlabel("recall (invasive)"); ax.set_ylabel("precision (invasive)"); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.set_title(f"binary invasive detection: AP = {m['bin_ap_invasive']:.3f}"); ax.legend(fontsize=7, loc="lower left")
    # per-class PR curves
    ax = axes[1]
    for c in range(N_INVASIVE):
        yc = (y == c)
        if yc.any():
            pc, rc, _ = precision_recall_curve(yc, probs[:, c]); ax.plot(rc, pc, color=C_BLUE, alpha=0.25, lw=1)
    ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.set_title(f"per-class PR, {N_INVASIVE} invasive classes: macro-AP = {m['macro_ap_invasive']:.3f}")
    # ROC
    ax = axes[2]
    if 0 < is_inv.sum() < len(y):
        fpr, tpr, _ = roc_curve(is_inv, score); ax.plot(fpr, tpr, color=C_BLUE, lw=2)
        fpr_t = float((~is_inv & (score >= t)).sum() / max((~is_inv).sum(), 1))
        ax.scatter([fpr_t], [m["bin_recall_invasive"]], color=C_ORANGE, s=40, zorder=3)
    ax.plot([0, 1], [0, 1], color=C_GRAY, ls=":", lw=1)
    ax.set_xlabel("false-positive rate (not_invasive flagged)"); ax.set_ylabel("true-positive rate (invasive found)")
    ax.set_title(f"ROC: AUROC = {m['bin_auroc_invasive']:.3f}")
    fig.suptitle(title); fig.tight_layout(); fig.savefig(out_dir / "pr_roc.png"); plt.show()


def plot_confusion(m: dict, out_dir: Path, title="", normalize=True):
    cm = np.array(m["confusion_matrix"], dtype=float)
    if normalize:
        cm = cm / np.clip(cm.sum(1, keepdims=True), 1, None)
    fig, ax = plt.subplots(figsize=(11, 10))
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1 if normalize else None)
    ax.set_xticks(range(len(ALL_LABELS))); ax.set_yticks(range(len(ALL_LABELS)))
    ax.set_xticklabels(ALL_LABELS, rotation=90, fontsize=6); ax.set_yticklabels(ALL_LABELS, fontsize=6)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.grid(False)
    ax.set_title(title or ("confusion matrix, row-normalised (recall on the diagonal)" if normalize else "confusion matrix (counts)"))
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02); fig.tight_layout(); fig.savefig(out_dir / "confusion_matrix.png"); plt.show()


def plot_per_class_f1(m: dict, out_dir: Path, support: pd.Series, title=""):
    f1 = pd.Series(m["per_class_f1"][:N_INVASIVE], index=INVASIVE_CLASSES).sort_values()
    fig, ax = plt.subplots(figsize=(7, 0.22 * N_INVASIVE + 1.5))
    ax.barh(f1.index, f1.values, color=C_BLUE, height=0.7)
    for i, (name, v) in enumerate(f1.items()):
        ax.text(min(v + 0.01, 0.9), i, f"n={int(support.get(name, 0))}", va="center", fontsize=6, color="#52514e")
    ax.set_xlim(0, 1); ax.set_xlabel("F1 on validation (at the tuned threshold)"); ax.tick_params(axis="y", labelsize=7)
    ax.set_title(title or f"per-class F1; macro-F1 (invasive) = {m['macro_f1_invasive']:.3f}")
    fig.tight_layout(); fig.savefig(out_dir / "per_class_f1.png"); plt.show()

In [ ]:
# (1) validation macro-F1 per epoch
fig, ax = plt.subplots(figsize=(7, 3.4))
for kind, c in zip(probes, [C_BLUE, C_ORANGE, C_AQUA]):
    h = probes[kind]["hist"]; ax.plot(h.epoch, h.macro_f1_invasive, color=c, lw=2, label=kind)
ax.axhline(m_zs["macro_f1_invasive"], color=C_GRAY, ls="--", lw=1); ax.text(0.2, m_zs["macro_f1_invasive"] + 0.005, "zero-shot", fontsize=7, color="#52514e")
ax.set_xlabel("epoch (0 = before training)"); ax.set_ylabel("val macro-F1 (invasive)"); ax.set_ylim(0, 1.02)
ax.set_title("linear heads on frozen BioCLIP 2 embeddings"); ax.legend(fontsize=7, frameon=False, loc="lower right")
fig.tight_layout(); fig.savefig(OUT_PROBE / "probe_curves.png"); plt.show()

# (2) WiSE interpolation
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(wise.alpha, wise.macro_f1_invasive, color=C_BLUE, lw=2, marker="o", ms=5, label="macro-F1 (invasive)")
ax.plot(wise.alpha, wise.top1_acc_all, color=C_ORANGE, lw=2, marker="o", ms=5, label="top-1 accuracy (41)")
ax.set_xlabel("alpha (0 = zero-shot text weights, 1 = trained head)"); ax.set_ylim(min(0.8, wise[["macro_f1_invasive", "top1_acc_all"]].values.min() - 0.02), 1.0)
ax.set_title("WiSE-FT interpolation of the species head (val)"); ax.legend(fontsize=7, frameon=False)
fig.tight_layout(); fig.savefig(OUT_PROBE / "wise_alpha.png"); plt.show()

# (3) all runs side by side, with the benchmark fine-tuned runs
rows_c = [{"run": f"{f.parent.parent.name} / {f.parent.name}", "family": "fine-tuned (team)",
           "macro_f1_invasive": json.load(open(f))["macro_f1_invasive"], "top1_acc_all": json.load(open(f))["top1_acc_all"]}
          for f in sorted(TEAM_RUNS.glob("*/*/metrics_final.json")) if f.parent.parent.name not in ("bioclip2_zeroshot", "bioclip2_probe")]
rows_c += [{"run": f"bioclip2 / {n}", "family": "BioCLIP 2 (this notebook)", "macro_f1_invasive": m["macro_f1_invasive"],
            "top1_acc_all": m["top1_acc_all"]} for n, (p, m) in results.items()]
allc = pd.DataFrame(rows_c).sort_values("macro_f1_invasive")
fig, axes = plt.subplots(1, 2, figsize=(12, 0.32 * len(allc) + 1.5), sharey=True)
for ax, col, lab_ in zip(axes, ["macro_f1_invasive", "top1_acc_all"], ["macro-F1 (invasive, primary)", "top-1 accuracy (41 labels, KPI)"]):
    ax.barh(allc.run, allc[col], height=0.6, color=[C_ORANGE if f_.startswith("BioCLIP") else C_BLUE for f_ in allc.family])
    for i, v in enumerate(allc[col]): ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=7, color="#52514e")
    ax.set_xlim(0, 1.12); ax.set_xlabel(lab_); ax.tick_params(axis="y", labelsize=7)
axes[0].legend(handles=[Patch(color=C_ORANGE, label="BioCLIP 2: zero-shot and frozen-feature heads"), Patch(color=C_BLUE, label="fine-tuned runs (team)")],
               fontsize=7, frameon=False, loc="lower right")
fig.suptitle("validation metrics, same split and same metric code", fontsize=9); fig.tight_layout()
fig.savefig(OUT_PROBE / "comparison_all_runs.png"); plt.show()

# (4) per-class F1: zero-shot vs the best head
best_name = cmp.drop(index=[i for i in cmp.index if i.startswith("zero_shot")]).macro_f1_invasive.idxmax()
f_zs = pd.Series(m_zs["per_class_f1"][:N_INVASIVE], index=INVASIVE_CLASSES)
f_b = pd.Series(results[best_name][1]["per_class_f1"][:N_INVASIVE], index=INVASIVE_CLASSES)
order = f_zs.sort_values().index
fig, ax = plt.subplots(figsize=(7, 0.22 * N_INVASIVE + 1.5))
yy = np.arange(len(order))
ax.hlines(yy, np.minimum(f_zs[order], f_b[order]), np.maximum(f_zs[order], f_b[order]), color=C_GRAY, lw=1)
ax.scatter(f_zs[order], yy, color=C_BLUE, s=22, zorder=3, label="zero-shot"); ax.scatter(f_b[order], yy, color=C_ORANGE, s=22, zorder=3, label=best_name)
ax.set_yticks(yy); ax.set_yticklabels(order, fontsize=7); ax.set_xlabel("per-class F1 on val"); ax.set_xlim(max(0, min(f_zs.min(), f_b.min()) - 0.05), 1.01)
ax.legend(fontsize=7, frameon=False, loc="lower right"); ax.set_title("per invasive class: zero-shot vs best frozen-feature head", fontsize=9)
fig.tight_layout(); fig.savefig(OUT_PROBE / "per_class_f1_zs_vs_best.png"); plt.show()

# (5) the template's own plots for the best head
bd = OUT_PROBE / best_name; pb, mb = results[best_name]
plot_threshold_sweep(mb["_sweep"], mb["threshold"], bd, f"bioclip2_probe / {best_name}\nthreshold sweep on validation")
plot_pr_roc(pb, y_val, mb, bd, f"bioclip2_probe / {best_name}")
plot_confusion(mb, bd, f"bioclip2_probe / {best_name}: confusion matrix (val, row-normalised)")

## D. Open set with the full regional vocabulary

Part A again, with the same removed species (same seeds), but with all extra species from Part B in the vocabulary, so unknown plants can land on a similar regional species instead of an invasive one. The extras are never removed: they stand in for the regional species the model would know in use. The threshold is tuned once with nothing removed, then held fixed. The 441-class scientific-prompt setting separates the prompt change from the vocabulary change. Needs Part B to have run (it uses `extra_txt` and `T_SCI`).

In [ ]:
# ---------- D. Part A again, with the full regional vocabulary (val only) ----------
# Same seeds as Part A, so the same background species are removed. Three settings:
#   <PROMPT>_441    : Part A as run (441 classes, threshold from notebook 14)
#   scientific_441  : same 441 classes, scientific prompts (separates the prompt change from the vocabulary change)
#   scientific_full : 441 + all extra species from Part B; threshold tuned once with nothing removed, then fixed
# Extras are never removed: they stand in for the regional species the model would know in deployment.
FULL_TXT = torch.cat([base_txt, extra_txt])
M_FULL = np.vstack([M, np.zeros((len(extra_txt), N_INVASIVE), np.float32)])
T_FULL = compute_metrics(probs_from_text(val_emb, FULL_TXT, M_FULL)[1], y_val, None, THRESHOLD_GRID)["threshold"]
SETTINGS = {f"{PROMPT}_441": (TEXT[PROMPT], M, T_ZS)} if PROMPT != "scientific" else {}
SETTINGS |= {"scientific_441": (base_txt, M, T_SCI), "scientific_full": (FULL_TXT, M_FULL, T_FULL)}
print({k: round(v[2], 2) for k, v in SETTINGS.items()}, "| full vocabulary size:", len(FULL_TXT))

rows_full, absorbed_full = [], pd.Series(dtype=float)
for name, (txt, Mm, t) in SETTINGS.items():
    for f in OPENSET_FRACS:
        for s in (range(OPENSET_SEEDS) if 0 < f < 1 else [0]):
            rng = np.random.default_rng(s)
            removed = rng.choice(BG, size=int(round(f * len(BG))), replace=False) if f > 0 else np.array([], dtype=int)
            keep = np.setdiff1d(np.arange(len(txt)), removed)
            p_voc, p_inv = probs_from_text(val_emb, txt[keep], Mm[keep])
            pred = predict_not_invasive(p_inv, t)
            unseen = np.isin(v_val, removed); seen_bg = ~is_inv_val & ~unseen
            flagged = pred < N_INVASIVE
            named_extra = keep[p_voc.argmax(1)] >= len(vocab)
            rows_full.append({"setting": name, "threshold": t, "frac_removed": f, "seed": s, "n_unseen_images": int(unseen.sum()),
                              "false_alarm_unseen": flagged[unseen].mean() if unseen.any() else np.nan,
                              "unseen_named_extra": named_extra[unseen].mean() if unseen.any() else np.nan,
                              "false_alarm_seen": flagged[seen_bg].mean() if seen_bg.any() else np.nan,
                              "invasive_recall": flagged[is_inv_val].mean(),
                              "macro_f1_invasive": macro_f1_invasive(y_val, pred), "top1_acc_all": float((pred == y_val).mean())})
            if name == "scientific_full" and f == 0.5:
                hit = pd.Series(np.array(ALL_LABELS)[pred[unseen & flagged]]).value_counts()
                absorbed_full = absorbed_full.add(hit, fill_value=0)
openset_full = pd.DataFrame(rows_full)
openset_full.to_csv(OUT_ROBUST / "openset_full_vocab.csv", index=False)
summary_full = openset_full.groupby(["setting", "frac_removed"]).mean(numeric_only=True).drop(columns="seed")
for col in ["false_alarm_unseen", "unseen_named_extra", "false_alarm_seen", "invasive_recall", "macro_f1_invasive"]:
    print(f"\n{col} (mean over draws)"); display(summary_full[col].unstack(0).round(4))

In [ ]:
g = openset_full.groupby(["setting", "frac_removed"])
mean, std = g.mean(numeric_only=True), g.std(numeric_only=True).fillna(0)
colors = dict(zip(SETTINGS, [C_BLUE, C_AQUA, C_ORANGE]))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for name in SETTINGS:
    mu, sd = mean.loc[name], std.loc[name]; x = mu.index.to_numpy() * 100
    axes[0].errorbar(x, mu.false_alarm_unseen * 100, yerr=sd.false_alarm_unseen * 100, color=colors[name], lw=2, marker="o", ms=5,
                     capsize=3, label=f"{name} (t = {SETTINGS[name][2]:.2f})")
    axes[1].errorbar(x, mu.macro_f1_invasive, yerr=sd.macro_f1_invasive, color=colors[name], lw=2, marker="o", ms=5, capsize=3, label=name)
axes[0].set_ylim(bottom=0); axes[0].set_ylabel("unknown species flagged invasive (%)")
axes[0].set_title("false alarms on unknown species"); axes[0].legend(fontsize=7, frameon=False)
axes[1].set_ylim(0, 1.02); axes[1].set_ylabel("macro-F1 (invasive, primary)")
axes[1].set_title("macro-F1 as background species are removed"); axes[1].legend(fontsize=7, frameon=False, loc="lower left")
for ax in axes: ax.set_xlabel("background species removed from the vocabulary (%)")
fig.suptitle(f"BioCLIP 2 zero-shot: does the full regional vocabulary absorb unknown species? (val only, mean ± sd over {OPENSET_SEEDS} draws)", fontsize=9)
fig.tight_layout(); fig.savefig(OUT_ROBUST / "openset_full_vocab.png"); plt.show()

s = (absorbed_full / OPENSET_SEEDS).sort_values().tail(12)
if not s.empty:
    fig, ax = plt.subplots(figsize=(6.5, 0.3 * len(s) + 1.2))
    ax.barh(s.index, s.values, color=C_ORANGE, height=0.6); ax.set_xlabel("unknown-species val images flagged as this class (per draw)")
    ax.tick_params(axis="y", labelsize=7); ax.set_title("where unknown species go, full vocabulary, 50% of background removed", fontsize=8)
    fig.tight_layout(); fig.savefig(OUT_ROBUST / "openset_full_vocab_absorbers_50.png"); plt.show()

## E. Better classifiers with the same 40-output labels

Same labels as the benchmark models: invasive photos carry their class, background photos are all `not_invasive`, no species information is used. Five classifiers on the frozen BioCLIP 2 embeddings: the linear head of Part C (reference), the benchmark one-vs-all logistic regression (standardised features, C swept), a 41-way softmax logistic regression, a one-hidden-layer network, and k-nearest neighbours. Needs sections 1-3 only. Hyperparameters and threshold are picked on val, so every number is slightly optimistic.

**Note on Part C:** the `bce40_random` linear head is under-trained. The embeddings are L2-normalised (each dimension is about 1/sqrt(768) = 0.04), and with that learning rate and number of epochs the head's scores stay small; its tuned threshold is 0.01. The standardised logistic regression below, with the same labels, is the fair estimate of what the 40-output setup reaches on these features.

In [ ]:
# ---------- E. Better classifiers with the same 40-output labels (no species information) ----------
# Needs sections 1-3 only (not A-D). Every variant sees exactly what the team's models see: invasive photos labelled with
# their class, background photos all labelled not_invasive. Hyperparameters (C, k, epoch) and the threshold are picked on val,
# like the template and the benchmark logistic-regression notebook (17_logreg_baselines.ipynb), so every number here is slightly optimistic.
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler

E_C_GRID = [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]       # logistic regression: inverse L2 strength
E_K_GRID = [1, 3, 5, 10, 20, 50]                    # k-nearest neighbours
E_MLP = dict(hidden=1024, dropout=0.3, lr=1e-3, wd=1e-2, epochs=60)

XE = torch.cat([tr_plain, tr_flip]); yE = np.tile(train_df.y.to_numpy(), 2)       # each image as is and mirrored (as in C)
T40 = np.eye(N_INVASIVE + 1, dtype=np.float32)[yE][:, :N_INVASIVE]                 # not_invasive rows are all zero
score = lambda p: compute_metrics(p, y_val, None, THRESHOLD_GRID)
runs_e, sweeps_e = {}, {}

def keep_best(name, cands, pname):
    # cands: list of (param, val probabilities for the 40 invasive classes)
    hist = pd.DataFrame([{pname: v, "macro_f1_invasive": score(p)["macro_f1_invasive"]} for v, p in cands])
    i = int(hist.macro_f1_invasive.idxmax()); v, p = cands[i]
    runs_e[name] = {"p": p, "m": score(p), "param": f"{pname}={v}"}; sweeps_e[name] = hist
    print(f"{name:>17}: best {pname} = {v} | macro-F1 {hist.macro_f1_invasive.max():.4f}")

# E0. linear layer + BCE, the same as C's bce40_random (should reproduce ~0.748): the reference for E1-E4
def train_torch(make_head, epochs, lr, wd):
    torch.manual_seed(SEED); head = make_head(); opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=wd)
    Tt, out = torch.tensor(T40), []
    for ep in range(1, epochs + 1):
        head.train(); perm = torch.randperm(len(XE))
        for i in range(0, len(XE), BATCH):
            b = perm[i:i + BATCH]; loss = F.binary_cross_entropy_with_logits(head(XE[b]), Tt[b])
            opt.zero_grad(); loss.backward(); opt.step()
        head.eval()
        with torch.no_grad():
            out.append((ep, torch.sigmoid(head(val_emb)).numpy()))
    return out

def linear_head():
    l = nn.Linear(XE.shape[1], N_INVASIVE); nn.init.xavier_uniform_(l.weight); nn.init.zeros_(l.bias); return l
t0 = time.time(); keep_best("linear_bce40", train_torch(linear_head, 40, 1e-3, 1e-4), "epoch"); print(f"  {time.time() - t0:.0f}s")

# E1. logistic regression, one per invasive class (the benchmark recipe: standardised features, one-vs-rest, C sweep)
scaler = StandardScaler().fit(XE.numpy()); Xs, Vs = scaler.transform(XE.numpy()), scaler.transform(val_emb.numpy())
def fit_quiet(clf, X, y):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning); return clf.fit(X, y)
t0 = time.time()
keep_best("logreg_ovr40", [(C, fit_quiet(OneVsRestClassifier(LogisticRegression(C=C, max_iter=2000), n_jobs=-1), Xs, T40).predict_proba(Vs))
                           for C in E_C_GRID], "C"); print(f"  {time.time() - t0:.0f}s")

# E2. one softmax logistic regression over 41 labels: not_invasive becomes a class of its own instead of "all zeros"
assert len(np.unique(yE)) == N_INVASIVE + 1
t0 = time.time()
keep_best("logreg_softmax41", [(C, fit_quiet(LogisticRegression(C=C, max_iter=2000), Xs, yE).predict_proba(Vs)[:, :N_INVASIVE])
                               for C in E_C_GRID], "C"); print(f"  {time.time() - t0:.0f}s")

# E3. small neural network (one hidden layer): can carve the mixed not_invasive group into several regions, which a linear layer cannot
def mlp_head():
    return nn.Sequential(nn.Linear(XE.shape[1], E_MLP["hidden"]), nn.GELU(), nn.Dropout(E_MLP["dropout"]), nn.Linear(E_MLP["hidden"], N_INVASIVE))
t0 = time.time(); keep_best("mlp_bce40", train_torch(mlp_head, E_MLP["epochs"], E_MLP["lr"], E_MLP["wd"]), "epoch"); print(f"  {time.time() - t0:.0f}s")

# E4. k-nearest neighbours (cosine similarity, similarity-weighted vote over the 41 labels). It needs no species labels, but a
# background photo's neighbours are usually photos of its own species, so it keeps species structure implicitly.
tr_emb = F.normalize(tr_plain + tr_flip, dim=-1) if TTA_HFLIP else tr_plain                 # same treatment as val_emb
top_s, top_i = (val_emb @ tr_emb.T).topk(max(E_K_GRID), dim=1)
ytr = train_df.y.to_numpy(); rows_idx = np.arange(len(y_val))[:, None]
def knn_probs(k, tau=0.05):
    w = torch.softmax(top_s[:, :k] / tau, dim=1).numpy(); lab = ytr[top_i[:, :k].numpy()]
    p = np.zeros((len(y_val), N_INVASIVE + 1)); np.add.at(p, (np.repeat(rows_idx, k, 1), lab), w)
    return p[:, :N_INVASIVE]
keep_best("knn41", [(k, knn_probs(k)) for k in E_K_GRID], "k")

In [ ]:
# Compare with zero-shot and with the linear head (paired bootstrap on val), save each variant like a template run, plot.
ref = runs_e["linear_bce40"]["m"]["_pred"]; rows_e = []
for name, r in runs_e.items():
    m = r["m"]; d = OUT_PROBE / f"e_{name}"; d.mkdir(parents=True, exist_ok=True)
    json.dump({"model": "bioclip2_probe", "iteration": f"e_{name}", "selected": r["param"], "tta_hflip": TTA_HFLIP, **scalars(m),
               "per_class_f1": dict(zip(ALL_LABELS, m["per_class_f1"])), "per_class_ap": dict(zip(INVASIVE_CLASSES, m["per_class_ap"])),
               "labels": ALL_LABELS, "confusion_matrix": m["confusion_matrix"]}, open(d / "metrics_final.json", "w"), indent=1, default=float)
    pd.DataFrame({"observation_id": val_df.observation_id, "label": val_df.label, "group": val_df.group, "scientific_name": val_df.scientific_name,
                  "pred": [ALL_LABELS[i] for i in m["_pred"]], "max_prob": r["p"].max(1)}).to_csv(d / "val_predictions.csv", index=False)
    zl, _, zh = paired_bootstrap(y_val, m_zs["_pred"], m["_pred"])
    ll, _, lh = paired_bootstrap(y_val, ref, m["_pred"]) if name != "linear_bce40" else (0, 0, 0)
    rows_e.append({"run": name, "selected": r["param"], **{k: m[k] for k in ["macro_f1_invasive", "top1_acc_all", "bin_precision_invasive",
                   "bin_recall_invasive", "threshold"]},
                   "delta_vs_linear": m["macro_f1_invasive"] - runs_e["linear_bce40"]["m"]["macro_f1_invasive"], "dl_ci_low": ll, "dl_ci_high": lh,
                   "delta_vs_zero_shot": m["macro_f1_invasive"] - m_zs["macro_f1_invasive"], "dz_ci_low": zl, "dz_ci_high": zh})
cmp_e = pd.DataFrame(rows_e).set_index("run"); cmp_e.to_csv(OUT_PROBE / "part_e_comparison.csv")
display(cmp_e.round(4))
print("Intervals are 95% paired-bootstrap intervals of the difference; one that contains 0 means within val noise.")

# reference lines: zero-shot, species-level head from C (if saved), the benchmark DINOv3 logistic regression (if present)
refs = {"zero-shot": m_zs["macro_f1_invasive"]}
for lab_, f in [("species head (C)", OUT_PROBE / "species_random" / "metrics_final.json"),
                ("DINOv3 logistic regression (benchmark)", TEAM_RUNS / "logreg_dinov3_vits16" / "00_baseline" / "metrics_final.json")]:
    if f.exists(): refs[lab_] = json.load(open(f))["macro_f1_invasive"]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1.1, 1]})
ax = axes[0]; order = cmp_e.macro_f1_invasive.sort_values().index
ax.barh(order, cmp_e.loc[order, "macro_f1_invasive"], height=0.6, color=C_BLUE)
for i, v in enumerate(cmp_e.loc[order, "macro_f1_invasive"]): ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=7, color="#52514e")
for (lab_, v), c in zip(refs.items(), [C_GRAY, C_AQUA, C_ORANGE]):
    ax.axvline(v, color=c, ls="--", lw=1.2, label=f"{lab_}: {v:.3f}")
ax.set_xlim(0, 1.1); ax.set_xlabel("val macro-F1 (invasive)"); ax.tick_params(axis="y", labelsize=8)
ax.set_title("40-output labels on frozen BioCLIP 2 features", fontsize=9); ax.legend(fontsize=7, frameon=False, loc="lower right")
ax = axes[1]
for name, c in zip(["logreg_ovr40", "logreg_softmax41", "knn41"], [C_BLUE, C_AQUA, C_ORANGE]):
    h = sweeps_e[name]
    ax.plot(h.iloc[:, 0].astype(float), h.macro_f1_invasive, color=c, lw=2, marker="o", ms=4, label=f"{name} (x = {h.columns[0]})")
ax.set_xscale("log"); ax.set_ylim(0, 1.02); ax.set_xlabel("C (logistic regression) or k (neighbours), log scale"); ax.set_ylabel("val macro-F1")
ax.set_title("hyperparameter sweeps", fontsize=9); ax.legend(fontsize=7, frameon=False, loc="lower right")
fig.tight_layout(); fig.savefig(OUT_PROBE / "part_e_classifiers.png"); plt.show()

## How to read the results
- **A (open set):** the orange curve is the field-relevant number: the share of images of species the model has never been told about that it flags as invasive. A flat, low curve means BioCLIP's "not invasive" decision survives unknown species; a steep curve means the 0.97 depends on the closed vocabulary. The absorber bars show which invasive classes act as catch-alls (usually look-alikes of common natives).
- **B (bigger vocabulary):** if macro-F1 drops a lot as the vocabulary grows, part of the 0.97 came from choosing among few options. The right panel shows how often val images are pulled towards species that are not in the dataset.
- **C (heads):** compare each head with zero-shot through the bootstrap interval, not the point value: with ~24 val images per invasive class, differences below ~0.01–0.02 are noise. The best-epoch selection on val (as in the template) makes trained heads look slightly better than they are.
- **D (full vocabulary):** compare the unknown-species false-alarm curves of the three settings at the same removed share. A flat, low curve for the full vocabulary means the regional names absorb unknown species; the cost is a lower threshold and more false alarms on known background species.
- **E (classifiers):** compare each variant with the linear head and with zero-shot through the bootstrap intervals. The benchmark logistic regression is the like-for-like comparison with the other backbones' probes.
- Nothing here touches `test.csv` or `test_unseen.csv`. Note that `test_unseen` is meant to measure unknown species; it stays valid because its 30 hold-out classes are never added to any vocabulary here.